# 01 — Problema e benchmark

## Objetivo

Qual referência simples deve orientar as próximas etapas?

Comparamos Regressão Logística e Random Forest na mesma validação. O teste fica
reservado até o notebook final.

In [ ]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import time
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.auxiliares import COLUNAS_NOMINAIS, carregar_base_preparada
from src.visual_utils import grafico_comparacao_modelos

dados = carregar_base_preparada(RAIZ)

### Visão rápida da base

In [ ]:
dados = carregar_base_preparada(RAIZ)

display(dados.head())

resumo_base = pd.Series({
    "registros": len(dados),
    "variaveis": dados.shape[1],
    "valores_ausentes": int(dados.isna().sum().sum()),
    "duplicatas_exatas": int(dados.duplicated().sum()),
    "taxa_inadimplencia": f"{dados['inadimplente'].mean():.2%}",
})

display(resumo_base.to_frame("resultado"))

variaveis_exemplo = [
    "limite_credito",
    "idade",
    "valor_fatura_set",
    "valor_pago_set",
]

display(
    dados[variaveis_exemplo]
    .agg(["min", "median", "max"])
    .T
)

## 1.1 — Como separar features, target e conjuntos?

Agora que a divisão está definida, salvamos os três conjuntos. Assim, os
próximos experimentos usam exatamente os mesmos dados sem repetir o split.

In [ ]:
pasta_processados = RAIZ / "data" / "processed"
pasta_processados.mkdir(parents=True, exist_ok=True)

dados_treino = X_treino.copy()
dados_treino["inadimplente"] = y_treino

dados_validacao = X_validacao.copy()
dados_validacao["inadimplente"] = y_validacao

dados_teste = X_teste.copy()
dados_teste["inadimplente"] = y_teste

dados_treino.to_csv(pasta_processados / "treino.csv", index=False)
dados_validacao.to_csv(pasta_processados / "validacao.csv", index=False)
dados_teste.to_csv(pasta_processados / "teste.csv", index=False)

pd.DataFrame({
    "conjunto": ["treino", "validação", "teste"],
    "linhas": [len(dados_treino), len(dados_validacao), len(dados_teste)],
    "proporcao": [len(dados_treino), len(dados_validacao), len(dados_teste)],
    "taxa_inadimplencia": [y_treino.mean(), y_validacao.mean(), y_teste.mean()],
}).assign(proporcao=lambda tabela: tabela["proporcao"] / tabela["linhas"].sum())

O split estratificado produz 60% para treino, 20% para validação e 20% para
teste. ID e target não entram nas 23 features.

## 1.2 — O que a Regressão Logística entrega?

In [ ]:
pd.DataFrame({
    "classe_prevista": previsoes_logisticas[:5],
    "probabilidade": probabilidades_logisticas[:5],
})

In [ ]:
print(f"Average Precision: {ap_logistica:.3f}")

## 1.3 — O bagging melhora a referência?

## 1.4 — Como comparar os dois modelos?

In [ ]:
fig = grafico_comparacao_modelos(resultados)
fig.show()

## 1.5 — Resultado

A Random Forest melhora o ranking probabilístico em relação à logística e será
o exemplo prático de bagging. Precision, Recall e F1 ainda usam limiar 0,50.